# Assignment 1 – Problem 3
## Satellite Image Classification (Urban vs. Natural) Using Classical Vision + ML  
### Classical Computer Vision + Machine Learning
**Course:** Computer Vision (S2-25_AIMLCZG525)

---

### Goal 
Build a classical machine-learning system that classifies satellite images into Urban or 
Natural categories using handcrafted image features. The system must incorporate pixel
level, texture, and gradient-based features and evaluate performance on a real external 
image. 

### Implementation Overview
1. Dataset Preparation (fMoW dataset – Urban vs Natural, ~500 each)
2. Preprocessing (resize, convert to grayscale, CLAHE, Gaussian blur)
3. Feature Engineering (Pixel intensity histogram, LBP texture descriptor, Edges using Sobel/Canny, Histogram of Oriented Gradients (HOG))
4. Feature Selection (PCA)
5. Model Training (SVM + Random Forest, 5-fold CV)
6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. External Image Prediction
8. Analysis & Discussion

## Import Required Libraries

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
import warnings
import pickle
#warnings.filterwarnings('ignore')

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import kagglehub

# ── Image Processing ──────────────────────────────────────────────────────────
import cv2                                      # OpenCV for image I/O and processing
from skimage.feature import hog, local_binary_pattern  # HOG and LBP
from skimage import exposure

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✅ All libraries imported successfully.")
print(f"   OpenCV   : {cv2.__version__}")
print(f"   NumPy    : {np.__version__}")
print(f"   Pandas   : {pd.__version__}")

## 1. Dataset Preparation
**fMoW Dataset – Urban vs Natural (~500 images each)**

- Download and organize fMoW dataset images into two classes: Urban and Natural
- Limit to ~500 images per class for balanced training
- Split into train/validation/test sets

In [10]:
# 1. Dataset Preparation
EXTERNAL_IMG   = 'external_test.jpg'   # your own real-world image
IMG_SIZE       = (128, 128)
MAX_PER_CLASS  = 500             # cap to keep runtime manageable
FEATURES_FILE  = 'features.pkl'  # cached features

# Define urban and natural classes for EuroSAT
# Mapping EuroSAT classes to broader Urban/Natural categories.
# EuroSAT classes generally include: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture,
# PermanentCrop, Residential, River, SeaLake.
URBAN_EUROSAT_CATEGORIES = ['Residential', 'Industrial', 'Highway']
NATURAL_EUROSAT_CATEGORIES = ['Forest', 'River', 'SeaLake', 'Pasture', 'PermanentCrop', 'HerbaceousVegetation', 'AnnualCrop']

# Download the EuroSAT dataset from Kaggle
print("Downloading EuroSAT dataset from Kaggle... This may take a moment.")
# The path returned by dataset_download is typically the base directory of the extracted dataset
try:
    eurosat_path = kagglehub.dataset_download("waseemalastal/eurosat-rgb-dataset")
    print(f"EuroSAT dataset downloaded to: {eurosat_path}")
except Exception as e:
    print(f"Error downloading from Kaggle: {e}")
    print("Please ensure you have authenticated Kaggle API if necessary (e.g., using `kaggle.json`).")
    eurosat_path = None # Set to None to prevent further processing if download fails

all_images = []
all_labels = []

urban_count = 0
natural_count = 0

if eurosat_path:
    # Iterate through the downloaded dataset directory to collect images
    # Assuming the structure is eurosat_path/class_name/image.jpg
    for root, dirs, files in os.walk(eurosat_path):
        # Extract the class name from the current directory
        # The class folder is usually directly under the main dataset folder
        if root == eurosat_path:
            continue # Skip the root directory itself, which only contains class directories

        class_name = os.path.basename(root)
        print(f"Processing class folder: {class_name}")

        target_label = None
        if class_name in URBAN_EUROSAT_CATEGORIES:
            target_label = 'urban'
        elif class_name in NATURAL_EUROSAT_CATEGORIES:
            target_label = 'natural'
        else:
            print(f"  Skipping class '{class_name}' as it's not mapped to urban/natural.")
            continue

        # Check if we still need images for this target label
        if target_label == 'urban' and urban_count >= MAX_PER_CLASS:
            print(f"  Already collected {MAX_PER_CLASS} images for 'urban'. Skipping this class.")
            continue
        if target_label == 'natural' and natural_count >= MAX_PER_CLASS:
            print(f"  Already collected {MAX_PER_CLASS} images for 'natural'. Skipping this class.")
            continue

        for file_name in files:
            if urban_count >= MAX_PER_CLASS and natural_count >= MAX_PER_CLASS:
                break # Collected enough images for both categories overall

            # Only process image files (e.g., .jpg, .png)
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_path = os.path.join(root, file_name)
                img_cv2 = cv2.imread(image_path)

                if img_cv2 is None:
                    print(f"  Warning: Could not read image {image_path}. Skipping.")
                    continue

                # cv2.imread reads as BGR by default for color images. If it's grayscale, it's 2D.
                # Convert to BGR if needed (e.g., if it's grayscale or RGBA, for consistency).
                if img_cv2.ndim == 2: # Grayscale image
                    img_cv2 = cv2.cvtColor(img_cv2, cv2.COLOR_GRAY2BGR)
                elif img_cv2.shape[2] == 4: # RGBA image
                    img_cv2 = cv2.cvtColor(img_cv2, cv2.COLOR_RGBA2BGR)


                if target_label == 'urban' and urban_count < MAX_PER_CLASS:
                    all_images.append(img_cv2)
                    all_labels.append('urban')
                    urban_count += 1
                elif target_label == 'natural' and natural_count < MAX_PER_CLASS:
                    all_images.append(img_cv2)
                    all_labels.append('natural')
                    natural_count += 1

        if urban_count >= MAX_PER_CLASS and natural_count >= MAX_PER_CLASS:
            print(f"  Reached MAX_PER_CLASS for both 'urban' ({urban_count}) and 'natural' ({natural_count}). Stopping collection.")
            break # Exit outer loop if enough images are collected
else:
    print("Dataset download failed or path not found. Cannot proceed with image loading.")

print(f"\nTotal images loaded : {len(all_images)}")
print(f"  Urban   : {all_labels.count('urban')}")
print(f"  Natural : {all_labels.count('natural')}")

Error downloading from Kaggle: name 'kagglehub' is not defined
Please ensure you have authenticated Kaggle API if necessary (e.g., using `kaggle.json`).
Dataset download failed or path not found. Cannot proceed with image loading.

Total images loaded : 0
  Urban   : 0
  Natural : 0


## 2. Preprocessing
**Steps: Resize → Grayscale → CLAHE → Gaussian Blur**

- Resize all images to a fixed resolution (e.g., 128×128)
- Convert to grayscale for uniform feature extraction
- Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
- Apply Gaussian blur to reduce noise

In [3]:
# 2. Preprocessing


## 3. Feature Engineering
**Descriptors: Pixel Intensity Histogram | LBP | Sobel/Canny Edges | HOG**

- **Pixel Intensity Histogram**: Distribution of pixel values per channel
- **LBP (Local Binary Pattern)**: Texture descriptor capturing local structure
- **Edges (Sobel / Canny)**: Edge magnitude maps for structural features
- **HOG (Histogram of Oriented Gradients)**: Shape and gradient orientation features

In [4]:
# 3. Feature Engineering


## 4. Feature Selection
**Method: Principal Component Analysis (PCA)**

- Concatenate all engineered features into a single feature vector
- Apply PCA to reduce dimensionality while retaining variance (e.g., 95%)
- Visualize explained variance ratio

In [5]:
# 4. Feature Selection (PCA)


## 5. Model Training
**Classifiers: SVM + Random Forest | Validation: 5-Fold Cross-Validation**

- Train Support Vector Machine (SVM) with RBF kernel on PCA-reduced features
- Train Random Forest classifier with tuned hyperparameters
- Apply 5-fold cross-validation to assess generalization performance

In [6]:
# 5. Model Training (SVM + Random Forest, 5-Fold CV)


## 6. Evaluation
**Metrics: Accuracy | Precision | Recall | F1-Score | Confusion Matrix**

- Compute Accuracy, Precision, Recall, and F1-Score for each model
- Plot Confusion Matrix for SVM and Random Forest
- Compare performance of both classifiers

In [7]:
# 6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)


## 7. External Image Prediction
**Predict class (Urban / Natural) for a new unseen image**

- Load an external satellite image not seen during training
- Apply the same preprocessing and feature extraction pipeline
- Predict and display the class label using the best trained model

In [8]:
# 7. External Image Prediction


## 8. Analysis & Discussion
**Observations, Insights, and Conclusions**

- Discuss which features contributed most to classification performance
- Compare SVM vs Random Forest: strengths and weaknesses
- Analyze where the model succeeds and where it fails (error analysis)
- Suggest possible improvements (e.g., deep features, data augmentation)
- Final conclusions and takeaways

In [9]:
# 8. Analysis & Discussion
